# Standardized Multi-Organism 3D Part Segmentation: SDF + Multi-View SAM3
### Robust, Species-Agnostic Anatomical Fin & Appendage Extraction (Dorsal, Side, Tail, Body)

This notebook implements the **Standardized Hybrid 3D Part Segmentation** architecture combining:
1. **3D Shape Diameter Function (SDF)**: View-invariant geometric thickness analysis on the surface manifold.
2. **Mesh Surface Graph Clustering**: Connected component candidate proposal on thin geometric extremities.
3. **Multi-View SAM3 Vision-Language Prompt Voting**: Zero-shot semantic identity voting across 20 orthogonal/perspective views (`tail`, `top fin`, `side fins`).
4. **Species-Agnostic 3D Spatial Disambiguation**:
   - **Pectoral (Side) Fins**: Primary lateral appendages on the flanks (highest lateral span $|Z|$ and elevation $Y$).
   - **Dorsal (Top) Fin**: Median fin along the top ridge ($cy > 0.08, |cz| < 0.15$).
   - **Caudal (Tail) Fin**: Posterior terminal fin blade ($cx \ge 0.65$ or furthest posterior cluster).
   - **Main Body**: Seamlessly retains the head, trunk, belly, and pelvic base geometry without disconnected fragments.
5. **Morphological Gap Filling & Boundary Closure**: Topological majority-voting closure on the face adjacency graph.
6. **1-Round Fin Growth Extension (Boundary Dilation)**: Expands ONLY fin appendages into neighboring boundary faces to seal boundary seams.
7. **Disconnected Graph Island Removal**: Retains the dominant connected component per fin and merges all orphan fragments back into `Main Body`.
8. **Constrained Delaunay Hole Capping**: Capping open root boundary loops with CDT on isolated fin submeshes.
9. **Per-Model GPU VRAM Management**: Accelerated FP16/BF16 inference with automatic memory deallocation.

In [12]:
import os
import sys
import gc
from pathlib import Path
from collections import defaultdict
import numpy as np
import cv2
import torch
import trimesh
import networkx as nx
import triangle
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from transformers import Sam3Processor, Sam3Model

# Add repository root to path
sys.path.append("../..")
load_dotenv()

from animgen.core.models.model import BaseModelClass
from animgen.utils.mesh import triangle_areas
from animgen.rigging.shape_diameter_function import shape_diameter_function

## 1. Organism Definitions & Parameters

In [13]:
MODEL_PATH = "facebook/sam3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {DEVICE}")

ORGANISMS = {
    "tuna_dec": {
        "name": "Tuna (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Tuna.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/tuna_dec"),
    },
    "mackeral_dec": {
        "name": "Mackeral (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Mackeral.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/mackeral_dec"),
    },
    "shark_dec": {
        "name": "Shark (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Shark.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/shark_dec"),
    },
    "killer_whale_dec": {
        "name": "Killer Whale (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Killer_Whale.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/killer_whale_dec"),
    },
    "goldfish_dec": {
        "name": "Goldfish (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Goldfish.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/goldfish_dec"),
    },
    "dolphin_dec": {
        "name": "Dolphin (Decimated)",
        "mesh_path": Path("../../generated_data/models/models_backup_3/dec_mesh_Dolphin.glb"),
        "output_dir": Path("../../generated_data/test/test_segmented_hybrid/dolphin_dec"),
    },
}

PROMPTS = [
    "Tail fin",
    "Top fin",
    "Side fins",
]

Compute device: cuda


## 2. Planar Hole Capping via Constrained Delaunay Triangulation (CDT)

In [14]:
def cap_root_hole_with_triangle(submesh: trimesh.Trimesh) -> trimesh.Trimesh:
    submesh = submesh.copy()
    edges = submesh.faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2)
    edges_sorted = np.sort(edges, axis=1)
    unique_edges, unique_inverse, counts = np.unique(
        edges_sorted, axis=0, return_inverse=True, return_counts=True
    )
    boundary_edge_mask = counts[unique_inverse] == 1
    boundary_directed = edges[boundary_edge_mask]
    
    if len(boundary_directed) == 0:
        return submesh
        
    G = nx.DiGraph()
    for u, v in boundary_directed:
        G.add_edge(u, v)
        
    loops = [c for c in nx.simple_cycles(G) if len(c) >= 3]
    if not loops:
        return submesh
        
    all_verts = list(submesh.vertices)
    all_faces = list(submesh.faces)
    
    for loop_vert_indices in loops:
        loop_vert_indices = np.array(loop_vert_indices, dtype=np.int64)
        loop_pts_3d = submesh.vertices[loop_vert_indices]
        K = len(loop_pts_3d)
        if K < 3:
            continue
            
        centroid = loop_pts_3d.mean(axis=0)
        centered = loop_pts_3d - centroid
        _, _, vh = np.linalg.svd(centered)
        u1, u2 = vh[0], vh[1]
        normal = np.cross(u1, u2)
        norm_len = np.linalg.norm(normal)
        if norm_len > 1e-12:
            normal = normal / norm_len
            
        pts_2d = np.column_stack([np.dot(centered, u1), np.dot(centered, u2)])
        segments = np.column_stack([np.arange(K), np.roll(np.arange(K), -1)])
        
        try:
            tri_out = triangle.triangulate({'vertices': pts_2d, 'segments': segments}, 'p')
        except Exception:
            try:
                tri_out = triangle.triangulate({'vertices': pts_2d, 'segments': segments}, 'c')
            except Exception:
                continue
            
        out_verts_2d = tri_out['vertices']
        out_triangles = tri_out['triangles']
        
        vert_map = {i: loop_vert_indices[i] for i in range(K)}
        for i in range(K, len(out_verts_2d)):
            p2 = out_verts_2d[i]
            v3d = centroid + p2[0] * u1 + p2[1] * u2
            vert_map[i] = len(all_verts)
            all_verts.append(v3d)
            
        for tri in out_triangles:
            mapped_tri = [vert_map[tri[0]], vert_map[tri[1]], vert_map[tri[2]]]
            p0 = all_verts[mapped_tri[0]]
            p1 = all_verts[mapped_tri[1]]
            p2 = all_verts[mapped_tri[2]]
            tri_norm = np.cross(p1 - p0, p2 - p0)
            if np.dot(tri_norm, normal) < 0:
                mapped_tri = [mapped_tri[0], mapped_tri[2], mapped_tri[1]]
            all_faces.append(mapped_tri)
            
    return trimesh.Trimesh(vertices=np.array(all_verts), faces=np.array(all_faces), process=True)

## 3. Morphological Gap Filling, Boundary Growth Extension & Island Removal

In [15]:
def fill_face_gaps(mesh: trimesh.Trimesh, face_labels: np.ndarray, adj_dict: dict, max_iters: int = 2) -> np.ndarray:
    """Fills isolated face gaps using majority-voting across neighbor faces."""
    labels = face_labels.copy()
    for _ in range(max_iters):
        changed = 0
        for f in range(len(mesh.faces)):
            curr_lbl = labels[f]
            neighbors = adj_dict[f]
            if not neighbors:
                continue
            nb_labels = [labels[nb] for nb in neighbors]
            majority_lbl = max(set(nb_labels), key=nb_labels.count)
            if nb_labels.count(majority_lbl) >= len(neighbors) * 0.7 and majority_lbl != curr_lbl:
                labels[f] = majority_lbl
                changed += 1
        if changed == 0:
            break
    return labels


def expand_fin_boundaries(mesh: trimesh.Trimesh, face_labels: np.ndarray, adj_dict: dict, rounds: int = 1) -> np.ndarray:
    """
    Expands ONLY non-body appendage labels (fins) outward by N hops across face adjacency.
    Body faces bordering a fin are absorbed into that fin if >= half of their neighbors share that fin label.
    """
    labels = face_labels.copy()
    for _ in range(rounds):
        new_labels = labels.copy()
        body_faces = np.where(labels == 0)[0]
        for f in body_faces:
            fin_neighbors = [labels[nb] for nb in adj_dict[f] if labels[nb] != 0]
            if not fin_neighbors:
                continue
            # Most common adjacent fin label
            candidate_label = max(set(fin_neighbors), key=fin_neighbors.count)
            # Expand ONLY if at least 2 neighbors or >= half of all neighbors share this fin label
            if fin_neighbors.count(candidate_label) >= 2 or fin_neighbors.count(candidate_label) >= len(adj_dict[f]) * 0.5:
                new_labels[f] = candidate_label
        labels = new_labels
    return labels


def remove_orphan_islands(mesh: trimesh.Trimesh, face_labels: np.ndarray, adj_dict: dict, min_area_ratio: float = 0.08) -> np.ndarray:
    """
    For every fin appendage (label != 0), finds all connected components on the face adjacency graph.
    Retains ONLY the largest/dominant connected component per fin.
    All smaller disconnected orphan islands are merged directly back into Main Body (label = 0).
    """
    cleaned_labels = face_labels.copy()
    face_areas = triangle_areas(mesh.vertices, mesh.faces)
    
    unique_labels = np.unique(face_labels)
    for lbl in unique_labels:
        if lbl == 0:  # Skip Main Body
            continue
            
        lbl_faces = np.where(cleaned_labels == lbl)[0]
        if len(lbl_faces) == 0:
            continue
            
        visited = set()
        components = []
        lbl_set = set(lbl_faces)
        
        for f in lbl_faces:
            if f in visited:
                continue
            comp = []
            queue = [f]
            visited.add(f)
            while queue:
                curr = queue.pop()
                comp.append(curr)
                for nb in adj_dict[curr]:
                    if nb in lbl_set and nb not in visited:
                        visited.add(nb)
                        queue.append(nb)
            components.append(np.array(comp, dtype=np.int32))
            
        # Sort components by total surface area descending
        components.sort(key=lambda c: np.sum(face_areas[c]), reverse=True)
        
        # Primary component
        primary_area = np.sum(face_areas[components[0]])
        
        # Reassign small orphan islands back to Body (0)
        reassigned_count = 0
        for comp in components[1:]:
            comp_area = np.sum(face_areas[comp])
            if comp_area < primary_area * min_area_ratio:
                cleaned_labels[comp] = 0
                reassigned_count += len(comp)
        if reassigned_count > 0:
            print(f"   [Island Removal] Label {lbl}: reassigned {reassigned_count} orphan faces back to Main Body.")
            
    return cleaned_labels


## 4. Per-Model SAM3 Multi-View Inference (with GPU VRAM Release)

In [16]:
from animgen.rigging.SAM3 import SAM3Segmentation
from animgen.rigging.backproject import backproject_masks_to_faces

def run_sam3_multi_view(mesh_model: BaseModelClass, prompts: list[str] = PROMPTS):
    """Runs accelerated multi-view SAM3 segmentation and backprojects votes to 3D mesh faces."""
    with SAM3Segmentation(prompts=prompts) as sam3:
        masks_dict = sam3(mesh_model, threshold=0.5, mask_threshold=0.5)
        face_prompt_detected = backproject_masks_to_faces(
            masks_dict,
            mesh_model.views_output["faces"],
            len(mesh_model.mesh.faces)
        )
    return face_prompt_detected

## 5. Species-Agnostic Anatomical Classification & Segmentation Pipeline

In [17]:
def classify_fish_appendages(mesh: trimesh.Trimesh, raw_clusters: list, total_mesh_area: float, face_prompt_detected: dict):
    """
    Universal 4-part anatomical fin classification pipeline:
    1. Tail Fin (Caudal): Posterior terminal blade (cx >= 0.65, or cx > 0.58 on midline).
    2. Top Fin (Dorsal): Dorsal midline ridge (cy > 0.08, |cz| < 0.15).
    3. Pectoral Fins (Side Fins): Primary lateral flank pair (highest lateral span |Z| and elevation Y).
    4. Ventral / Belly regions & Keel: Seamlessly retained in Main Body.
    """
    face_areas = triangle_areas(mesh.vertices, mesh.faces)
    MAJOR_AREA_THRESHOLD = 0.0035 # 0.35% minimum mesh area
    
    cluster_meta = []
    for comp in raw_clusters:
        c_area = np.sum(face_areas[comp])
        pct = (c_area / total_mesh_area) * 100
        if pct < MAJOR_AREA_THRESHOLD:
            continue
        verts = np.unique(mesh.faces[comp])
        cent = mesh.vertices[verts].mean(axis=0)
        max_abs_z = np.max(np.abs(mesh.vertices[verts, 2]))
        
        prompt_votes = {p: int(np.sum(face_prompt_detected[p][comp])) for p in PROMPTS}
        top_prompt = max(prompt_votes, key=prompt_votes.get) if max(prompt_votes.values()) > 0 else "None"
        
        cluster_meta.append({
            "faces": comp,
            "area_pct": pct,
            "centroid": cent,
            "max_abs_z": max_abs_z,
            "top_prompt": top_prompt,
            "prompt_votes": prompt_votes,
            "label": None,
        })
        
    # Pass 1: Median & Terminal Fins (Tail & Dorsal) + Keel Anomaly Filter
    for c in cluster_meta:
        cx, cy, cz = c["centroid"]
        
        # Keel / Peduncle ridge anomaly filter
        if 0.45 <= cx < 0.70 and abs(cz) < 0.06 and cy < -0.12:
            print(f"   [Keel Anomaly Filtered] {c['area_pct']:5.2f}% area at x={cx:.2f}, y={cy:.2f} -> Retained in Body")
            c["label"] = "Main Body"
            continue
            
        # Tail Fin: extreme posterior
        if cx >= 0.65 or (cx > 0.58 and abs(cz) < 0.06 and abs(cy) < 0.15):
            c["label"] = "Tail Fin"
        # Dorsal Fin: top midline
        elif cy > 0.08 and abs(cz) < 0.15:
            c["label"] = "Top Fin"
            
    # Pass 2: Bilateral Paired Pectoral Fins (Side Fins)
    unlabeled = [c for c in cluster_meta if c["label"] is None]
    
    left_candidates = [c for c in unlabeled if c["centroid"][2] > 0.02]
    right_candidates = [c for c in unlabeled if c["centroid"][2] < -0.02]
    
    left_candidates.sort(key=lambda c: (c["max_abs_z"], c["centroid"][1]), reverse=True)
    right_candidates.sort(key=lambda c: (c["max_abs_z"], c["centroid"][1]), reverse=True)
    
    # Primary flank pair = Pectoral Fins; Secondary ventral clusters merge into Main Body
    for i, c in enumerate(left_candidates):
        if i == 0 and (c["max_abs_z"] > 0.12 or c["centroid"][1] >= -0.15):
            c["label"] = "Left Pectoral Fin"
        else:
            c["label"] = "Main Body"
            
    for i, c in enumerate(right_candidates):
        if i == 0 and (c["max_abs_z"] > 0.12 or c["centroid"][1] >= -0.15):
            c["label"] = "Right Pectoral Fin"
        else:
            c["label"] = "Main Body"
            
    classified = defaultdict(list)
    for c in cluster_meta:
        if c["label"] and c["label"] != "Main Body":
            print(f"   Classified {c['label']:20s} ({c['area_pct']:5.2f}% area) | cent=({c['centroid'][0]:+.2f}, {c['centroid'][1]:+.2f}, {c['centroid'][2]:+.2f}) | SAM: '{c['top_prompt']}'")
            classified[c["label"]].append(c["faces"])
            
    return classified

def segment_single_organism(org_key: str, org_info: dict, sdf_threshold: float = 0.30):
    print(f"\n{'='*60}")
    print(f"Processing Organism: {org_info['name']}")
    print(f"Mesh path: {org_info['mesh_path']}")
    print(f"{'='*60}")
    
    mesh_model = BaseModelClass(org_info["mesh_path"], renderer_size=(512, 512))
    trimesh_obj = mesh_model.mesh
    num_faces = len(trimesh_obj.faces)
    num_verts = len(trimesh_obj.vertices)
    face_areas = triangle_areas(trimesh_obj.vertices, trimesh_obj.faces)
    total_mesh_area = np.sum(face_areas)
    
    # 1. Shape Diameter Function
    print("1. Computing 3D Shape Diameter Function (SDF)...")
    sdf = shape_diameter_function(trimesh_obj, norm=True)
    print(f"   SDF computed: min={sdf.min():.3f}, max={sdf.max():.3f}, mean={sdf.mean():.3f}")
    
    # 2. SAM3 multi-view inference
    print("2. Running SAM3 Multi-View Inference...")
    face_prompt_detected = run_sam3_multi_view(mesh_model)
    total_sam_votes = np.sum(list(face_prompt_detected.values()), axis=0)
    
    # 3. Adjacency Graph Setup
    adj = trimesh_obj.face_adjacency
    adj_dict = defaultdict(list)
    for f1, f2 in adj:
        adj_dict[f1].append(f2)
        adj_dict[f2].append(f1)
        
    # 4. Thin Geometric Filtering & Snout Exclusion
    is_thin = (sdf < sdf_threshold)
    is_sam_candidate = (total_sam_votes >= 2)
    face_centroids = trimesh_obj.triangles.mean(axis=1)
    
    is_snout = (face_centroids[:, 0] < -0.75) & (np.abs(face_centroids[:, 2]) < 0.12) & (np.abs(face_centroids[:, 1]) < 0.12)
    hybrid_candidate = (is_thin | is_sam_candidate) & (~is_snout)
    
    visited = np.zeros(num_faces, dtype=bool)
    raw_clusters = []
    for f in np.where(hybrid_candidate)[0]:
        if visited[f]:
            continue
        comp = []
        queue = [f]
        visited[f] = True
        while queue:
            curr = queue.pop()
            comp.append(curr)
            for nb in adj_dict[curr]:
                if hybrid_candidate[nb] and not visited[nb]:
                    visited[nb] = True
                    queue.append(nb)
        raw_clusters.append(np.array(comp, dtype=np.int32))
        
    raw_clusters.sort(key=lambda c: np.sum(face_areas[c]), reverse=True)
    print(f"3. Found {len(raw_clusters)} geometric candidate clusters.")
    
    # 5. Anatomical Classification
    print("4. Applying Standardized Morphological Classification...")
    classified_appendages = classify_fish_appendages(trimesh_obj, raw_clusters, total_mesh_area, face_prompt_detected)
    
    # 6. Part Assembly & Morphological Gap Filling
    face_label_array = np.zeros(num_faces, dtype=np.int32)
    label_to_id = {"Main Body": 0}
    id_to_label = {0: "Main Body"}
    
    part_id = 1
    for label, comp_list in classified_appendages.items():
        label_to_id[label] = part_id
        id_to_label[part_id] = label
        for comp in comp_list:
            face_label_array[comp] = part_id
        part_id += 1
        
    print("5. Applying Morphological Gap Filling across Face Boundaries...")
    refined_face_labels = fill_face_gaps(trimesh_obj, face_label_array, adj_dict, max_iters=2)
    
    print("6. Applying 1-Round Growth Extension to Fin Boundary Faces...")
    expanded_face_labels = expand_fin_boundaries(trimesh_obj, refined_face_labels, adj_dict, rounds=1)
    
    print("7. Applying Disconnected Graph Island Removal (Retaining Dominant Component)...")
    final_face_labels = remove_orphan_islands(trimesh_obj, expanded_face_labels, adj_dict, min_area_ratio=0.08)
    
    final_appendages = {}
    for pid, label in id_to_label.items():
        if pid == 0:
            continue
        p_faces = np.where(final_face_labels == pid)[0]
        if len(p_faces) == 0:
            continue
        final_appendages[label] = {
            "faces": p_faces,
            "area": np.sum(face_areas[p_faces]),
            "centroid": trimesh_obj.vertices[np.unique(trimesh_obj.faces[p_faces])].mean(axis=0),
            "verts": np.unique(trimesh_obj.faces[p_faces]),
        }
        
    # Main Body
    body_faces = np.where(final_face_labels == 0)[0]
    final_appendages["Main Body"] = {
        "faces": body_faces,
        "area": np.sum(face_areas[body_faces]),
        "centroid": trimesh_obj.vertices[np.unique(trimesh_obj.faces[body_faces])].mean(axis=0),
        "verts": np.unique(trimesh_obj.faces[body_faces]),
    }
    
    # 7. Submesh Export with Planar Hole Capping
    out_dir = org_info["output_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    
    def get_filename(lbl):
        l_lower = lbl.lower()
        if "body" in l_lower:
            return "body.glb"
        elif "tail" in l_lower or "caudal" in l_lower:
            return "tail.glb"
        elif "top" in l_lower or "dorsal" in l_lower:
            return "top_fins.glb"
        elif "left" in l_lower and "pectoral" in l_lower:
            return "left_pectoral_fin.glb"
        elif "right" in l_lower and "pectoral" in l_lower:
            return "right_pectoral_fin.glb"
        else:
            return f"{l_lower.replace(' ', '_')}.glb"
            
    print(f"\n6. Exporting Watertight Submeshes to {out_dir}:")
    exported_summary = {}
    for label, data in final_appendages.items():
        blob_faces = data["faces"]
        subm = trimesh_obj.submesh([blob_faces], append=True)
        if "Main Body" not in label:
            capped_subm = cap_root_hole_with_triangle(subm)
        else:
            capped_subm = subm
            
        fname = get_filename(label)
        out_path = out_dir / fname
        capped_subm.export(out_path)
        
        area_pct = (data["area"] / total_mesh_area) * 100
        print(f"   {label:25s} -> {fname:22s} | {len(capped_subm.vertices):5d} verts, {len(capped_subm.faces):5d} faces | {area_pct:5.2f}% area")
        exported_summary[label] = {
            "file": fname,
            "verts": len(capped_subm.vertices),
            "faces": len(capped_subm.faces),
            "area_pct": area_pct,
        }
        
    return exported_summary, sdf, final_appendages, trimesh_obj

## 6. Execution across All 5 Decimated Organisms

In [18]:
all_summaries = {}
all_mesh_data = {}

for org_key, org_info in ORGANISMS.items():
    summary, sdf_vals, appendages, mesh_obj = segment_single_organism(org_key, org_info)
    all_summaries[org_key] = summary
    all_mesh_data[org_key] = {
        "sdf": sdf_vals,
        "mesh": mesh_obj,
        "appendages": appendages
    }

print("\n" + "="*70)
print("ALL 5 DECIMATED ORGANISMS SEGMENTED SUCCESSFULLY!")
print("="*70)


Processing Organism: Tuna (Decimated)
Mesh path: ../../generated_data/models/models_backup_3/dec_mesh_Tuna.glb
1. Computing 3D Shape Diameter Function (SDF)...
   SDF computed: min=0.000, max=1.000, mean=0.537
2. Running SAM3 Multi-View Inference...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

Rendering Multiviews...: 100%|██████████| 20/20 [00:01<00:00, 17.90it/s]


3. Found 104 geometric candidate clusters.
4. Applying Standardized Morphological Classification...
   Classified Tail Fin             ( 6.05% area) | cent=(+0.84, +0.02, -0.00) | SAM: 'Tail fin'
   Classified Top Fin              ( 5.12% area) | cent=(+0.01, +0.28, -0.00) | SAM: 'Top fin'
   Classified Left Pectoral Fin    ( 3.19% area) | cent=(-0.17, +0.03, +0.17) | SAM: 'Top fin'
   Classified Right Pectoral Fin   ( 3.03% area) | cent=(-0.17, +0.03, -0.17) | SAM: 'Top fin'
   Classified Tail Fin             ( 0.07% area) | cent=(+0.67, +0.06, -0.00) | SAM: 'None'
   Classified Tail Fin             ( 0.06% area) | cent=(+0.62, +0.07, -0.00) | SAM: 'None'
   Classified Tail Fin             ( 0.06% area) | cent=(+0.67, -0.03, -0.00) | SAM: 'None'
   Classified Tail Fin             ( 0.05% area) | cent=(+0.62, -0.04, -0.00) | SAM: 'None'
   Classified Top Fin              ( 0.05% area) | cent=(+0.38, +0.15, -0.00) | SAM: 'None'
   Classified Top Fin              ( 0.05% area) | cent=(+0

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

...

Rendering Multiviews...: 100%|██████████| 20/20 [00:01<00:00, 12.11it/s]


3. Found 79 geometric candidate clusters.
4. Applying Standardized Morphological Classification...
   Classified Tail Fin             (18.28% area) | cent=(+0.64, +0.08, +0.00) | SAM: 'Tail fin'
   Classified Top Fin              ( 7.60% area) | cent=(+0.10, +0.39, +0.00) | SAM: 'Top fin'
   Classified Left Pectoral Fin    ( 1.95% area) | cent=(-0.22, -0.09, +0.17) | SAM: 'Top fin'
   Classified Right Pectoral Fin   ( 1.94% area) | cent=(-0.21, -0.09, -0.17) | SAM: 'Top fin'
   Classified Top Fin              ( 0.02% area) | cent=(-0.62, +0.17, +0.00) | SAM: 'None'
   Classified Top Fin              ( 0.01% area) | cent=(+0.22, +0.18, -0.07) | SAM: 'Top fin'
   Classified Top Fin              ( 0.01% area) | cent=(+0.16, +0.21, -0.07) | SAM: 'Top fin'
   Classified Top Fin              ( 0.01% area) | cent=(+0.24, +0.15, -0.08) | SAM: 'Top fin'
   Classified Top Fin              ( 0.01% area) | cent=(+0.19, +0.20, -0.07) | SAM: 'Top fin'
   Classified Top Fin              ( 0.01% area)

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

Rendering Multiviews...: 100%|██████████| 20/20 [00:01<00:00, 14.69it/s]


3. Found 102 geometric candidate clusters.
4. Applying Standardized Morphological Classification...
   [Keel Anomaly Filtered]  0.00% area at x=0.69, y=-0.13 -> Retained in Body
   Classified Tail Fin             (11.01% area) | cent=(+0.84, -0.16, -0.00) | SAM: 'Tail fin'
   Classified Left Pectoral Fin    ( 3.78% area) | cent=(-0.34, -0.08, +0.17) | SAM: 'Side fins'
   Classified Right Pectoral Fin   ( 3.76% area) | cent=(-0.34, -0.08, -0.17) | SAM: 'Side fins'
   Classified Top Fin              ( 3.47% area) | cent=(+0.09, +0.35, +0.00) | SAM: 'Top fin'
   Classified Tail Fin             ( 0.03% area) | cent=(+0.69, -0.08, -0.04) | SAM: 'Tail fin'
   Classified Top Fin              ( 0.02% area) | cent=(-0.13, +0.12, +0.13) | SAM: 'Top fin'
   Classified Tail Fin             ( 0.02% area) | cent=(+0.68, -0.07, +0.04) | SAM: 'Tail fin'
   Classified Top Fin              ( 0.01% area) | cent=(-0.02, +0.18, +0.11) | SAM: 'Top fin'
   Classified Top Fin              ( 0.01% area) | cent

## 7. SDF Distribution Analysis across Morphologies

## 8. 3D Color-Mapped Anatomical Part Visualization

In [19]:
COLOR_MAP = {
    "Main Body": [75, 85, 95, 255],
    "Top Fin": [0, 200, 255, 255],
    "Tail Fin": [255, 60, 60, 255],
    "Left Pectoral Fin": [0, 230, 118, 255],
    "Right Pectoral Fin": [255, 180, 0, 255],
    "Anal Fin": [200, 0, 255, 255],
    "Left Pelvic Fin": [100, 255, 200, 255],
    "Right Pelvic Fin": [255, 220, 100, 255],
}

for org_key, data in all_mesh_data.items():
    mesh = data["mesh"].copy()
    face_colors = np.zeros((len(mesh.faces), 4), dtype=np.uint8)
    face_colors[:] = COLOR_MAP["Main Body"]
    
    for label, app_data in data["appendages"].items():
        c = COLOR_MAP.get(label, [200, 200, 200, 255])
        face_colors[app_data["faces"]] = c
        
    mesh.visual.face_colors = face_colors
    print(f"Rendered color-coded submesh for {ORGANISMS[org_key]['name']}.")
    # To view in interactive 3D window:
    # mesh.show()

Rendered color-coded submesh for Tuna (Decimated).
Rendered color-coded submesh for Mackeral (Decimated).
Rendered color-coded submesh for Shark (Decimated).
Rendered color-coded submesh for Killer Whale (Decimated).
Rendered color-coded submesh for Goldfish (Decimated).
Rendered color-coded submesh for Dolphin (Decimated).


## 9. Comprehensive Segmentation Results Table

In [20]:
rows = []
for org_key, parts in all_summaries.items():
    org_name = ORGANISMS[org_key]["name"]
    for part_name, info in parts.items():
        rows.append({
            "Organism": org_name,
            "Anatomical Part": part_name,
            "Filename": info["file"],
            "Vertices": info["verts"],
            "Faces": info["faces"],
            "Area %": f"{info['area_pct']:.2f}%"
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

                Organism    Anatomical Part               Filename  Vertices  Faces Area %
        Tuna (Decimated)           Tail Fin               tail.glb      1007   2010  6.17%
        Tuna (Decimated)            Top Fin           top_fins.glb       989   1974  5.20%
        Tuna (Decimated)  Left Pectoral Fin  left_pectoral_fin.glb       555   1110  3.22%
        Tuna (Decimated) Right Pectoral Fin right_pectoral_fin.glb       545   1084  3.06%
        Tuna (Decimated)          Main Body               body.glb      7710  15161 82.36%
    Mackeral (Decimated)           Tail Fin               tail.glb      1477   2948 10.94%
    Mackeral (Decimated)            Top Fin           top_fins.glb       665   1320  5.20%
    Mackeral (Decimated) Right Pectoral Fin right_pectoral_fin.glb       234    462  1.23%
    Mackeral (Decimated)  Left Pectoral Fin  left_pectoral_fin.glb       189    370  1.06%
    Mackeral (Decimated)          Main Body               body.glb      5806  11371 81.57%